In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()
\n# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")
@show Metal.devices()


# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))


include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))

using .commonBatchs, .flexOPT, .planet1D, .GeoPoints\n

In [ ]:
famousEquationType="2DacousticTime"

config= (
    name = "OPT3",
    orderBtime = 1,
    orderBspace = 1,
    pointsInSpace = 3,
    pointsInTime = 3,
    supplementaryOrder = 2,
    fieldItpl = (
        ptsSpace = 1, ptsTime = 1,
        offsetSpace = 1.0, offsetTime = 1,
        YorderBspace = -2, YorderBtime = 1,
    ),
    materItpl = (
        ptsSpace = 3, ptsTime = 1,
        offsetSpace = 0, offsetTime = 1,
        YorderBspace = -1, YorderBtime = 1,
    ),
)

In [ ]:
modelName="authors"
imageFile="../dataInput/model/moi/authors.png"
modelDefinitionMethod="2DimageFile" # ToyModel or 2DimageFile (or 1DsphericalPlanet)
model=defineModel(imageFile);
#
#boxGridsMarmousi = constructLocalBox(model,-3000.0,0.0,0.0,9200.0)
boxGrids = lazyProduceOrLoad("AuthorsCoordInfo",constructLocalBox,model,-3000.0,0.0,0.0,9200.0)
#seismicModelMarmousi = makeAdHocSeismicModel(model, 1.0, 2.8, 1.5, 5.5, 0.0, 3.2)
seismicModel=lazyProduceOrLoad("seismicModelAuthors",makeAdHocSeismicModel,model, 1.0, 2.8, 1.5, 5.5, 0.0, 3.2)

#constructLocalBox for marmousi models should be written!
using CairoMakie
xvals = [p.xz[1] for p in boxGrids.allGridsInCartesian[:,1]]*1.e-3
zvals = [p.xz[2] for p in boxGrids.allGridsInCartesian[1,:]]*1.e-3
fig, ax, hm = heatmap(
    #topo.x,topo.y,topo.z';
    #collect((0:1:(Nx-1)).*Δx).*1.e-3,(collect(0:1:(Nz-1)).*Δz.+altMin).*1.e-3, seismicModel.ρ;
    xvals, zvals, seismicModel.Vsh;
    colormap = :BrBG,
    #colorrange=(0,4),
    axis = (aspect = DataAspect(), xlabel = "horizontal", ylabel = "depth", title = "Vsh model")
)
Colorbar(fig[1,2], hm, label="Vsh")
fig

In [ ]:
imagefile = "../dataInput/model/moi/authors.png"
colormap = "hot" #colormap can be RGB vector or predefined colormap

floatMatrix=read2DimageModel(imagefile,colormap;min=1000,max=3300, showRecoveredImage=true) 
@show size(floatMatrix)

# Brocher (2005)
function vp_from_rho(rho::Float64)
    return rho*1.5
end

function vs_from_vp(vp::Float64)
    return vp/1.7
end

# this is only for Lyon concours use!

data=floatMatrix
@show maximum(data),minimum(data)
  # Convert to Float32 (single precision)
data_single = Float32.(data)

# Flatten in column-major order (Fortran-style)
data_vec = vec(data_single)

# Write to binary file
open("GCNF.rho", "w") do io
    write(io, data_vec)
end



data=vp_from_rho.(data)
@show maximum(data),minimum(data)
# Convert to Float32 (single precision)
data_single = Float32.(data)

# Flatten in column-major order (Fortran-style)
data_vec = vec(data_single)

# Write to binary file
open("GCNF.vp", "w") do io
    write(io, data_vec)
end


data=vs_from_vp.(data)
@show maximum(data),minimum(data)
# Convert to Float32 (single precision)
data_single = Float32.(data)

# Flatten in column-major order (Fortran-style)
data_vec = vec(data_single)

# Write to binary file
open("GCNF.vs", "w") do io
    write(io, data_vec)
end

In [ ]:
Δnum = (boxGrids.Δx,boxGrids.Δz,1.0) 
modelAuth=(models=((seismicModel.Vsh)),modelName="authors",modelPoints=(size(seismicModel.Vsh)...,10))


In [ ]:
@unpack orderBtime, orderBspace, pointsInSpace, pointsInTime, supplementaryOrder, fieldItpl, materItpl = config


concreteParametersForOPTConstruction = @strdict famousEquationType Δ=Δnum orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl
#optRec = myProduceOrLoad(makeOPTsemiSymbolic, concreteParametersForOPTConstruction, "semiSymbolic")
makeOPTsemiSymbolic(concreteParametersForOPTConstruction)

In [ ]:
continue
params = @strdict optRec=optRec modelFam=modelAuth absorbingBoundaries=nothing maskedRegionInSpace=nothing
numOpt = numericalOperatorConstruction(params)

numOps = numOpt["numOperators"]